# Self-play benchmark — parallel vs vectorized (Option B)

Decides whether to flip a 9x9 run to `self_play_mode: "vectorized"`.

Run the cells top to bottom. **Cell 1 is the important one** — if it reports no
CUDA, stop: the numbers below would be measured on a CPU-only torch and say
nothing about this box's GPU behaviour.

Requires an idle GPU. Cell 2 checks.

## 1. Environment — is this kernel the one with CUDA torch?

`sys.executable` below is also the answer to "which interpreter has torch on
this box" — use that path from the shell if you want to run the CLI version.

In [ ]:
import os, sys, pathlib

# Repo root on sys.path, whether the kernel started here or in notebooks/.
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

print("repo root:", ROOT)
print("python   :", sys.executable)

from scripts.bench_self_play import describe_device, run_bench, DEFAULTS
DEVICE = describe_device("auto")

## 2. Is the GPU free?

The benchmark spawns many workers and saturates the GPU. Run it next to a live
training job and both engines are measured under contention — which penalises
the worker-pool path most, biasing the result toward vectorized. Any python
process listed here means something is already training.

In [ ]:
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

## 3. Smoke test (~1 min)

Tiny run, just to confirm both engines execute in this kernel. The ratio here is
meaningless — 5x5, 25 sims, no warmup convergence.

In [ ]:
_ = run_bench(games=4, sims=25, num_workers=4, max_moves=40, warmup_games=1)

## 4. The real measurement

Matches the 9x9 N=4 training config. Takes a while — 50 games at 800 sims,
three times per engine, plus warmup.

Lower `repeat` to 1 for a faster (noisier) answer.

In [ ]:
BENCH = dict(
    board_size=9, num_players=4, walls=10,
    num_channels=128, num_res_blocks=8,
    sims=800, games=50, vec_games=64,
    num_workers=32, batch_size=256,
    max_moves=300, explore_moves=15,
    device="auto", warmup_games=2, repeat=3,
)
results = run_bench(**BENCH)
results

## 5. Confirm the ordering effect is not the story

If step 4's margin is inside ~10%, re-run with the engines swapped. A result
that flips with order is not a real difference.

In [ ]:
results_swapped = run_bench(**{**BENCH, "order": "vectorized,parallel"})

for label, r in [("original", results), ("swapped", results_swapped)]:
    if {"parallel", "vectorized"} <= set(r):
        print(f"{label:9s}: parallel/vectorized = "
              f"{r['parallel'] / r['vectorized']:.2f}x")

## 6. Verdict

Flip the run to `"vectorized"` only if **both** orderings agree that vectorized
is faster, by a margin you would notice.

To switch (see the PR body for the full note):

1. Stop the run.
2. `configs/config_9x9.json` → `"self_play_mode": "vectorized"`, `"vec_games": 64`.
3. Resume — weights persist via `latest.pt`; only the replay buffer refills.

The games played after the switch differ from what the parallel engine would
have produced (different exploration-noise stream), but their quality does not:
the vectorized driver runs exact sequential MCTS with no virtual-loss
approximation.